# Nine-city India LST fine-tuning (Colab)

Cities: Bhopal, Bhubaneswar, Chandigarh, Jamshedpur, Mysuru, Visakhapatnam, Surat, Jaipur, Srinagar.

This notebook builds a quality-controlled multi-city dataset, uses date-separated train/validation/test splits, starts from the **original IBM weights**, and compares the fine-tuned model with the untouched baseline. GT is already in Celsius. The earlier Srinagar fine-tuned checkpoint is deliberately not used.

**Important limitations:** no notebook can guarantee a significant improvement or zero runtime errors before the actual Drive data and installed model package are tested. This notebook fails early on missing/ambiguous city paths and on checkpoint/model incompatibility rather than silently training a wrong model.

## 1. Mount Drive and set paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, re, json, math, random, warnings, subprocess, sys, copy
import numpy as np, pandas as pd

WORK = Path('/content/drive/MyDrive/colab_workspace')
DATA = WORK/'data'
SOURCE_CSV = WORK/'multi_city_data_audit.csv'
ORIGINAL_WEIGHTS = WORK/'models_and_configs'/'LST_model_weights_only.pth'
OUT = WORK/'india_nine_city_lst_experiment_v1'
ALIGNED, PATCHES, CKPTS = OUT/'aligned_scenes', OUT/'patches', OUT/'checkpoints'
for p in [OUT,ALIGNED,PATCHES,CKPTS]: p.mkdir(parents=True,exist_ok=True)

CITIES = ['bhopal_ind','bhubaneshwar_ind','chandigarh_ind','jamshedpur_ind',
          'mysuru_ind','visakhapatnam_ind','surat_ind','jaipur_ind','srinagar_ind']
GT_UNIT='C'  # already Celsius; do not convert
PATCH=128
STRIDE=128
MIN_SCENE_VALID=0.50
MIN_PATCH_VALID=0.80
SEED=42

assert SOURCE_CSV.exists(), SOURCE_CSV
assert ORIGINAL_WEIGHTS.exists(), ORIGINAL_WEIGHTS
print('Original IBM weights:', ORIGINAL_WEIGHTS)

Mounted at /content/drive
Original IBM weights: /content/drive/MyDrive/colab_workspace/models_and_configs/LST_model_weights_only.pth


## 2. Install dependencies and check GPU

In [2]:
%pip -q install rasterio pyproj terratorch lightning

import torch, rasterio, lightning
print('Torch:',torch.__version__,'CUDA:',torch.cuda.is_available())
print('Rasterio:',rasterio.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab before continuing.')
import terratorch

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Torch: 2.14.1+cu130 CUDA: True
Rasterio: 1.5.2


# **PROCESSING**

In [3]:
# NINE-CITY PATH SETUP

WORKSPACE = Path("/content/drive/MyDrive/colab_workspace")
DATA_ROOT = WORKSPACE / "data" / "hls"
MODEL_DIR = WORKSPACE / "models_and_configs"

CITIES = [
    "bhopal_ind",
    "bhubaneshwar_ind",
    "chandigarh_ind",
    "jamshedpur_ind",
    "mysuru_ind",
    "visakhapatnam_ind",
    "surat_ind",
    "jaipur_ind",
    "srinagar_ind",
]

# Original source directories; these are not aligned output directories.
CITY_DIRS = {
    city: {
        "hls": DATA_ROOT / city / "hls-predict-ready",
        "gt": DATA_ROOT / city / "target-lst",
    }
    for city in CITIES
}

print("Checking original HLS and target folders:\n")

missing = []

for city, paths in CITY_DIRS.items():
    hls_ok = paths["hls"].is_dir()
    gt_ok = paths["gt"].is_dir()

    print(f"{city}")
    print("  HLS:", paths["hls"], "->", hls_ok)
    print("  GT: ", paths["gt"], "->", gt_ok)

    if not hls_ok:
        missing.append((city, "HLS", paths["hls"]))
    if not gt_ok:
        missing.append((city, "GT", paths["gt"]))

if missing:
    print("\nMissing directories:")
    for city, kind, path in missing:
        print(city, kind, path)

    raise FileNotFoundError(
        "Some source directories were not found. "
        "Check their actual Drive locations before proceeding."
    )

print("\nAll nine HLS and target directories exist.")

Checking original HLS and target folders:

bhopal_ind
  HLS: /content/drive/MyDrive/colab_workspace/data/hls/bhopal_ind/hls-predict-ready -> True
  GT:  /content/drive/MyDrive/colab_workspace/data/hls/bhopal_ind/target-lst -> True
bhubaneshwar_ind
  HLS: /content/drive/MyDrive/colab_workspace/data/hls/bhubaneshwar_ind/hls-predict-ready -> True
  GT:  /content/drive/MyDrive/colab_workspace/data/hls/bhubaneshwar_ind/target-lst -> True
chandigarh_ind
  HLS: /content/drive/MyDrive/colab_workspace/data/hls/chandigarh_ind/hls-predict-ready -> True
  GT:  /content/drive/MyDrive/colab_workspace/data/hls/chandigarh_ind/target-lst -> True
jamshedpur_ind
  HLS: /content/drive/MyDrive/colab_workspace/data/hls/jamshedpur_ind/hls-predict-ready -> True
  GT:  /content/drive/MyDrive/colab_workspace/data/hls/jamshedpur_ind/target-lst -> True
mysuru_ind
  HLS: /content/drive/MyDrive/colab_workspace/data/hls/mysuru_ind/hls-predict-ready -> True
  GT:  /content/drive/MyDrive/colab_workspace/data/hls/mysur

In [6]:
# NINE-CITY OUTPUT STRUCTURE

EXPERIMENT_ROOT = WORKSPACE / "data" / "granite_nine_city_v1"

ALIGNED_ROOT = EXPERIMENT_ROOT / "aligned_targets"
PATCH_ROOT = EXPERIMENT_ROOT / "patch_dataset"

TRAIN_INPUT = PATCH_ROOT / "train" / "inputs"
TRAIN_TARGET = PATCH_ROOT / "train" / "targets"

VAL_INPUT = PATCH_ROOT / "val" / "inputs"
VAL_TARGET = PATCH_ROOT / "val" / "targets"

TEST_INPUT = PATCH_ROOT / "test" / "inputs"
TEST_TARGET = PATCH_ROOT / "test" / "targets"

for city in CITIES:
    (ALIGNED_ROOT / city).mkdir(parents=True, exist_ok=True)

for folder in [
    TRAIN_INPUT, TRAIN_TARGET,
    VAL_INPUT, VAL_TARGET,
    TEST_INPUT, TEST_TARGET,
]:
    folder.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = MODEL_DIR / "nine_city_checkpoints"
RUN_DIR = WORKSPACE / "runs" / "nine_city_lst"
LOG_DIR = WORKSPACE / "tb_logs"

FIT_CONFIG = MODEL_DIR / "nine_city_fit_config.yaml"

for folder in [CHECKPOINT_DIR, RUN_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("New experiment output root:", EXPERIMENT_ROOT)

New experiment output root: /content/drive/MyDrive/colab_workspace/data/granite_nine_city_v1


In [4]:
# # MATCH HLS AND GT FILES

# def get_tiffs(folder):
#     files = list(folder.glob("*.tif"))
#     files += list(folder.glob("*.tiff"))
#     return sorted(set(files))


# scene_pairs = []
# pair_report = []

# for city in CITIES:
#     hls_files = get_tiffs(CITY_DIRS[city]["hls"])
#     gt_files = get_tiffs(CITY_DIRS[city]["gt"])

#     gt_lookup = {p.name: p for p in gt_files}

#     matched = []
#     unmatched = []

#     for hls_path in hls_files:
#         expected_gt_name = hls_path.stem + ".lst.tif"
#         gt_path = gt_lookup.get(expected_gt_name)

#         if gt_path is None:
#             unmatched.append(hls_path.name)
#             continue

#         with rasterio.open(hls_path) as src:
#             if src.count != 7:
#                 raise ValueError(
#                     f"{city}/{hls_path.name}: expected 7 bands, "
#                     f"found {src.count}"
#                 )

#         matched.append((city, hls_path, gt_path))

#     scene_pairs.extend(matched)

#     pair_report.append({
#         "city": city,
#         "hls_files": len(hls_files),
#         "gt_files": len(gt_files),
#         "matched_pairs": len(matched),
#         "unmatched_hls": len(unmatched),
#     })

#     print(
#         f"{city}: {len(matched)} matched pairs "
#         f"from {len(hls_files)} HLS and {len(gt_files)} GT files"
#     )

#     if unmatched:
#         print("  Unmatched HLS examples:", unmatched[:5])

# pair_report = pd.DataFrame(pair_report)
# display(pair_report)

# pair_report.to_csv(
#     EXPERIMENT_ROOT / "scene_pairing_report.csv",
#     index=False
# )

# if not scene_pairs:
#     raise RuntimeError("No HLS/GT pairs were matched.")

# cities_with_pairs = {x[0] for x in scene_pairs}
# cities_without_pairs = sorted(set(CITIES) - cities_with_pairs)

# if cities_without_pairs:
#     raise RuntimeError(
#         "No matched scenes for these cities: "
#         + ", ".join(cities_without_pairs)
#         + ". Check the GT filename convention and source folders."
#     )

# print("Total matched scenes:", len(scene_pairs))

bhopal_ind: 44 matched pairs from 93 HLS and 61 GT files
  Unmatched HLS examples: ['bhopal_ind.T43QGF.20220114.T052040.tif', 'bhopal_ind.T43QGF.20220123.T051426.tif', 'bhopal_ind.T43QGF.20220130.T052034.tif', 'bhopal_ind.T43QGF.20220207.T052041.tif', 'bhopal_ind.T43QGF.20220208.T051423.tif']
bhubaneshwar_ind: 69 matched pairs from 90 HLS and 71 GT files
  Unmatched HLS examples: ['bhubaneshwar_ind.T45QUC.20220613.T044418.tif', 'bhubaneshwar_ind.T45QUC.20220621.T044355.tif', 'bhubaneshwar_ind.T45QUC.20220622.T043811.tif', 'bhubaneshwar_ind.T45QUC.20220731.T044434.tif', 'bhubaneshwar_ind.T45QUC.20220816.T044441.tif']
chandigarh_ind: 65 matched pairs from 69 HLS and 65 GT files
  Unmatched HLS examples: ['chandigarh_ind.T43RFP.20221004.T052514.tif', 'chandigarh_ind.T43RFP.20250419.T052407.tif', 'chandigarh_ind.T43RFP.20250724.T052430.tif', 'chandigarh_ind.T43RFQ.20230329.T052402.tif']
jamshedpur_ind: 70 matched pairs from 97 HLS and 70 GT files
  Unmatched HLS examples: ['jamshedpur_ind.

,city,hls_files,gt_files,matched_pairs,unmatched_hls
0,bhopal_ind,93,61,44,49
1,bhubaneshwar_ind,90,71,69,21
2,chandigarh_ind,69,65,65,4
3,jamshedpur_ind,97,70,70,27
4,mysuru_ind,46,37,37,9
5,visakhapatnam_ind,66,51,51,15
6,surat_ind,97,77,77,20
7,jaipur_ind,95,80,80,15
8,srinagar_ind,51,36,36,15


NameError: name 'EXPERIMENT_ROOT' is not defined

In [7]:
# # ALIGN TARGET LST RASTERS TO HLS GRIDS

# from rasterio.warp import reproject, Resampling
# from tqdm.auto import tqdm

# NODATA = -9999.0

# alignment_rows = []
# aligned_pairs = []

# for city, hls_path, gt_path in tqdm(
#     scene_pairs, desc="Aligning nine-city GT scenes"
# ):

#     out_dir = ALIGNED_ROOT / city
#     out_path = out_dir / f"{hls_path.stem}.lst.tif"

#     with rasterio.open(hls_path) as src, rasterio.open(gt_path) as tgt:

#         if src.crs is None or tgt.crs is None:
#             raise ValueError(
#                 f"Missing CRS: {city}, {hls_path.name}"
#             )

#         if src.count != 7 or tgt.count < 1:
#             raise ValueError(
#                 f"Unexpected band count: {city}, {hls_path.name}"
#             )

#         target_masked = tgt.read(1, masked=True)
#         target = target_masked.filled(np.nan).astype(np.float32)

#         # Also respect the GT raster's internal validity mask.
#         target_valid = (
#             np.isfinite(target)
#             & (tgt.read_masks(1) > 0)
#         )

#         if tgt.nodata is not None:
#             target_valid &= target != tgt.nodata

#         target[~target_valid] = np.nan

#         same_grid = (
#             src.width == tgt.width
#             and src.height == tgt.height
#             and src.crs == tgt.crs
#             and src.transform == tgt.transform
#         )

#         if same_grid:
#             aligned = target.copy()
#         else:
#             aligned = np.full(
#                 (src.height, src.width),
#                 np.nan,
#                 dtype=np.float32
#             )

#             reproject(
#                 source=target,
#                 destination=aligned,
#                 src_transform=tgt.transform,
#                 src_crs=tgt.crs,
#                 src_nodata=np.nan,
#                 dst_transform=src.transform,
#                 dst_crs=src.crs,
#                 dst_nodata=np.nan,
#                 resampling=Resampling.bilinear,
#                 init_dest_nodata=True,
#             )

#         valid = np.isfinite(aligned)
#         valid_fraction = float(valid.mean())

#         if not valid.any():
#             print("REJECTED — no target overlap:", city, hls_path.name)
#             continue

#         # GT is already in Celsius. No unit conversion or clipping.
#         values = aligned[valid]
#         p01, p50, p99 = np.percentile(values, [1, 50, 99])

#         if p01 < 0 or p99 > 65:
#             print(
#                 "CHECK GT RANGE:",
#                 city, hls_path.name,
#                 f"p01={p01:.2f}, median={p50:.2f}, p99={p99:.2f} °C"
#             )

#         aligned[~valid] = NODATA

#         profile = src.profile.copy()
#         profile.update(
#             driver="GTiff",
#             count=1,
#             dtype="float32",
#             nodata=NODATA,
#             compress="deflate",
#             predictor=3,
#         )
#         profile.pop("blockxsize", None)
#         profile.pop("blockysize", None)

#         with rasterio.open(out_path, "w", **profile) as dst:
#             dst.write(aligned, 1)
#             dst.set_band_description(1, "Aligned_LST_Celsius")

#         alignment_rows.append({
#             "city": city,
#             "hls": hls_path.name,
#             "gt": gt_path.name,
#             "aligned_gt": str(out_path),
#             "same_grid_before_alignment": same_grid,
#             "valid_fraction": valid_fraction,
#             "gt_p01_C": float(p01),
#             "gt_median_C": float(p50),
#             "gt_p99_C": float(p99),
#         })

#         aligned_pairs.append((city, hls_path, out_path))

# alignment_report = pd.DataFrame(alignment_rows)

# alignment_report.to_csv(
#     EXPERIMENT_ROOT / "alignment_report.csv",
#     index=False
# )

# display(
#     alignment_report.groupby("city").agg(
#         aligned_scenes=("hls", "count"),
#         median_valid_fraction=("valid_fraction", "median"),
#         min_valid_fraction=("valid_fraction", "min"),
#         median_GT_C=("gt_median_C", "median"),
#     )
# )

# print("Successfully aligned scenes:", len(aligned_pairs))

Aligning nine-city GT scenes:   0%|          | 0/529 [00:00<?, ?it/s]

CHECK GT RANGE: chandigarh_ind chandigarh_ind.T43RFP.20220419.T052425.tif p01=-0.95, median=39.02, p99=43.16 °C
CHECK GT RANGE: chandigarh_ind chandigarh_ind.T43RFP.20250411.T052430.tif p01=-7.31, median=22.54, p99=41.11 °C
CHECK GT RANGE: chandigarh_ind chandigarh_ind.T43RFQ.20230617.T052342.tif p01=23.36, median=44.99, p99=78.16 °C


KeyboardInterrupt: 

In [ ]:

# PERSISTENT HLS/GT MATCHING AND ALIGNMENT
# Safe to rerun after a Colab restart.

import os
import json
import hashlib
import numpy as np
import pandas as pd
import rasterio

from pathlib import Path
from rasterio.warp import reproject, Resampling
from tqdm.auto import tqdm

NODATA = -9999.0

# Persistent Drive locations
ALIGNED_ROOT = EXPERIMENT_ROOT / "aligned_targets"
MANIFEST_PATH = EXPERIMENT_ROOT / "alignment_manifest.json"
PAIR_REPORT_PATH = EXPERIMENT_ROOT / "scene_pairing_report.csv"
ALIGNMENT_REPORT_PATH = EXPERIMENT_ROOT / "alignment_report.csv"

EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
ALIGNED_ROOT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 1. LOAD EXISTING MANIFEST
# --------------------------------------------------

if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH, "r") as f:
        alignment_manifest = json.load(f)
    print(f"Loaded existing alignment manifest: {len(alignment_manifest)} entries")
else:
    alignment_manifest = {}
    print("No previous alignment manifest found. Starting fresh.")

# --------------------------------------------------
# 2. MATCH HLS AND GT FILES
# --------------------------------------------------

def get_tiffs(folder):
    return sorted(
        set(list(folder.glob("*.tif")) + list(folder.glob("*.tiff")))
    )

scene_pairs = []
pair_report_rows = []

for city in CITIES:
    hls_files = get_tiffs(CITY_DIRS[city]["hls"])
    gt_files = get_tiffs(CITY_DIRS[city]["gt"])

    gt_lookup = {p.name: p for p in gt_files}

    matched = []
    unmatched = []

    for hls_path in hls_files:
        expected_gt_name = hls_path.stem + ".lst.tif"
        gt_path = gt_lookup.get(expected_gt_name)

        if gt_path is None:
            unmatched.append(hls_path.name)
            continue

        with rasterio.open(hls_path) as src:
            if src.count != 7:
                raise ValueError(
                    f"{city}/{hls_path.name}: expected 7 bands, "
                    f"found {src.count}"
                )

        matched.append((city, hls_path, gt_path))

    scene_pairs.extend(matched)

    pair_report_rows.append({
        "city": city,
        "hls_files": len(hls_files),
        "gt_files": len(gt_files),
        "matched_pairs": len(matched),
        "unmatched_hls": len(unmatched),
    })

    print(
        f"{city}: {len(matched)} matched pairs "
        f"from {len(hls_files)} HLS and {len(gt_files)} GT files"
    )

    if unmatched:
        print("  Unmatched HLS examples:", unmatched[:5])

pair_report = pd.DataFrame(pair_report_rows)
display(pair_report)

pair_report.to_csv(PAIR_REPORT_PATH, index=False)

if not scene_pairs:
    raise RuntimeError("No HLS/GT pairs were matched.")

cities_with_pairs = {x[0] for x in scene_pairs}
cities_without_pairs = sorted(set(CITIES) - cities_with_pairs)

if cities_without_pairs:
    raise RuntimeError(
        "No matched scenes for: " + ", ".join(cities_without_pairs)
    )

print("Total matched scenes:", len(scene_pairs))

# --------------------------------------------------
# 3. ALIGN ONLY MISSING OR CHANGED SCENES
# --------------------------------------------------

def source_signature(path):
    """Detect changes to a source file."""
    stat = path.stat()
    return {
        "path": str(path.resolve()),
        "size": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
    }


def grid_signature(src):
    """Detect changes to the HLS output grid."""
    return {
        "width": src.width,
        "height": src.height,
        "crs": str(src.crs),
        "transform": list(src.transform),
        "count": src.count,
    }


def save_manifest():
    temp_path = MANIFEST_PATH.with_suffix(".tmp")
    with open(temp_path, "w") as f:
        json.dump(alignment_manifest, f, indent=2)
    os.replace(temp_path, MANIFEST_PATH)


alignment_rows = []
aligned_pairs = []

reused_count = 0
created_count = 0

for city, hls_path, gt_path in tqdm(
    scene_pairs, desc="Checking persistent GT alignment"
):

    out_dir = ALIGNED_ROOT / city
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"{hls_path.stem}.lst.tif"

    scene_id = f"{city}/{hls_path.name}"

    # Source metadata
    hls_signature = source_signature(hls_path)
    gt_signature = source_signature(gt_path)

    with rasterio.open(hls_path) as src:
        if src.crs is None:
            raise ValueError(f"Missing HLS CRS: {hls_path}")

        if src.count != 7:
            raise ValueError(f"Expected 7 HLS bands: {hls_path}")

        current_grid = grid_signature(src)

    old_entry = alignment_manifest.get(scene_id, {})

    cache_is_valid = (
        out_path.exists()
        and old_entry.get("hls_signature") == hls_signature
        and old_entry.get("gt_signature") == gt_signature
        and old_entry.get("grid_signature") == current_grid
        and old_entry.get("output_path") == str(out_path)
    )

    # Reuse the already aligned raster
    if cache_is_valid:
        with rasterio.open(out_path) as aligned_src:
            if (
                aligned_src.count == 1
                and aligned_src.width == current_grid["width"]
                and aligned_src.height == current_grid["height"]
            ):
                reused_count += 1

                aligned_pairs.append((city, hls_path, out_path))

                alignment_rows.append({
                    "city": city,
                    "hls": hls_path.name,
                    "gt": gt_path.name,
                    "aligned_gt": str(out_path),
                    "valid_fraction": old_entry.get("valid_fraction"),
                    "gt_p01_C": old_entry.get("gt_p01_C"),
                    "gt_median_C": old_entry.get("gt_median_C"),
                    "gt_p99_C": old_entry.get("gt_p99_C"),
                    "reused_from_cache": True,
                })

                continue

    # Cache missing or stale: perform alignment
    with rasterio.open(hls_path) as src, rasterio.open(gt_path) as tgt:

        if tgt.crs is None:
            raise ValueError(f"Missing GT CRS: {gt_path}")

        if tgt.count < 1:
            raise ValueError(f"GT has no raster bands: {gt_path}")

        target_masked = tgt.read(1, masked=True)
        target = target_masked.filled(np.nan).astype(np.float32)

        target_valid = (
            np.isfinite(target)
            & (tgt.read_masks(1) > 0)
        )

        if tgt.nodata is not None:
            target_valid &= target != tgt.nodata

        target[~target_valid] = np.nan

        same_grid = (
            src.width == tgt.width
            and src.height == tgt.height
            and src.crs == tgt.crs
            and src.transform == tgt.transform
        )

        if same_grid:
            aligned = target.copy()
        else:
            aligned = np.full(
                (src.height, src.width),
                np.nan,
                dtype=np.float32,
            )

            reproject(
                source=target,
                destination=aligned,
                src_transform=tgt.transform,
                src_crs=tgt.crs,
                src_nodata=np.nan,
                dst_transform=src.transform,
                dst_crs=src.crs,
                dst_nodata=np.nan,
                resampling=Resampling.bilinear,
                init_dest_nodata=True,
            )

        valid = np.isfinite(aligned)
        valid_fraction = float(valid.mean())

        if not valid.any():
            print("REJECTED — no target overlap:", scene_id)
            continue

        # GT is already Celsius. Do not convert or clip.
        values = aligned[valid]
        p01, p50, p99 = np.percentile(values, [1, 50, 99])

        if p01 < 0 or p99 > 65:
            print(
                "CHECK GT RANGE:",
                scene_id,
                f"p01={p01:.2f}, median={p50:.2f}, "
                f"p99={p99:.2f} °C"
            )

        aligned[~valid] = NODATA

        profile = src.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=NODATA,
            compress="deflate",
            predictor=3,
        )
        profile.pop("blockxsize", None)
        profile.pop("blockysize", None)

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(aligned, 1)
            dst.set_band_description(1, "Aligned_LST_Celsius")

        # Record successful alignment in persistent manifest
        entry = {
            "hls_signature": hls_signature,
            "gt_signature": gt_signature,
            "grid_signature": current_grid,
            "output_path": str(out_path),
            "valid_fraction": valid_fraction,
            "gt_p01_C": float(p01),
            "gt_median_C": float(p50),
            "gt_p99_C": float(p99),
        }

        alignment_manifest[scene_id] = entry
        save_manifest()

        aligned_pairs.append((city, hls_path, out_path))
        created_count += 1

        alignment_rows.append({
            "city": city,
            "hls": hls_path.name,
            "gt": gt_path.name,
            "aligned_gt": str(out_path),
            "valid_fraction": valid_fraction,
            "gt_p01_C": float(p01),
            "gt_median_C": float(p50),
            "gt_p99_C": float(p99),
            "reused_from_cache": False,
        })

# --------------------------------------------------
# 4. SAVE REPORTS
# --------------------------------------------------

alignment_report = pd.DataFrame(alignment_rows)

alignment_report.to_csv(
    ALIGNMENT_REPORT_PATH,
    index=False
)

save_manifest()

print("\nALIGNMENT SUMMARY")
print("Newly aligned:", created_count)
print("Reused from Drive:", reused_count)
print("Total usable aligned scenes:", len(aligned_pairs))

display(
    alignment_report.groupby("city").agg(
        aligned_scenes=("hls", "count"),
        median_valid_fraction=("valid_fraction", "median"),
        min_valid_fraction=("valid_fraction", "min"),
        median_GT_C=("gt_median_C", "median"),
    )
)

print("\nPersistent files:")
print("Aligned rasters:", ALIGNED_ROOT)
print("Manifest:", MANIFEST_PATH)
print("Pair report:", PAIR_REPORT_PATH)
print("Alignment report:", ALIGNMENT_REPORT_PATH)

if not aligned_pairs:
    raise RuntimeError("No aligned scenes are available.")


No previous alignment manifest found. Starting fresh.
bhopal_ind: 44 matched pairs from 93 HLS and 61 GT files
  Unmatched HLS examples: ['bhopal_ind.T43QGF.20220114.T052040.tif', 'bhopal_ind.T43QGF.20220123.T051426.tif', 'bhopal_ind.T43QGF.20220130.T052034.tif', 'bhopal_ind.T43QGF.20220207.T052041.tif', 'bhopal_ind.T43QGF.20220208.T051423.tif']
bhubaneshwar_ind: 69 matched pairs from 90 HLS and 71 GT files
  Unmatched HLS examples: ['bhubaneshwar_ind.T45QUC.20220613.T044418.tif', 'bhubaneshwar_ind.T45QUC.20220621.T044355.tif', 'bhubaneshwar_ind.T45QUC.20220622.T043811.tif', 'bhubaneshwar_ind.T45QUC.20220731.T044434.tif', 'bhubaneshwar_ind.T45QUC.20220816.T044441.tif']
chandigarh_ind: 65 matched pairs from 69 HLS and 65 GT files
  Unmatched HLS examples: ['chandigarh_ind.T43RFP.20221004.T052514.tif', 'chandigarh_ind.T43RFP.20250419.T052407.tif', 'chandigarh_ind.T43RFP.20250724.T052430.tif', 'chandigarh_ind.T43RFQ.20230329.T052402.tif']
jamshedpur_ind: 70 matched pairs from 97 HLS and 7

,city,hls_files,gt_files,matched_pairs,unmatched_hls
0,bhopal_ind,93,61,44,49
1,bhubaneshwar_ind,90,71,69,21
2,chandigarh_ind,69,65,65,4
3,jamshedpur_ind,97,70,70,27
4,mysuru_ind,46,37,37,9
5,visakhapatnam_ind,66,51,51,15
6,surat_ind,97,77,77,20
7,jaipur_ind,95,80,80,15
8,srinagar_ind,51,36,36,15


Total matched scenes: 529


Checking persistent GT alignment:   0%|          | 0/529 [00:00<?, ?it/s]

CHECK GT RANGE: chandigarh_ind/chandigarh_ind.T43RFP.20220419.T052425.tif p01=-0.95, median=39.02, p99=43.16 °C
CHECK GT RANGE: chandigarh_ind/chandigarh_ind.T43RFP.20250411.T052430.tif p01=-7.31, median=22.54, p99=41.11 °C
CHECK GT RANGE: chandigarh_ind/chandigarh_ind.T43RFQ.20230617.T052342.tif p01=23.36, median=44.99, p99=78.16 °C
CHECK GT RANGE: visakhapatnam_ind/visakhapatnam_ind.T44QQE.20230802.T045051.tif p01=-55.50, median=-46.09, p99=-41.87 °C


In [ ]:
# DATE-BASED SPLIT, WITHIN EACH CITY

import re
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)

def get_acquisition_date(path):
    match = re.search(r"20\d{6}", Path(path).stem)
    if match is None:
        return None
    return match.group(0)

dated_pairs = []

for city, hls_path, aligned_path in aligned_pairs:
    date = get_acquisition_date(hls_path)

    if date is None:
        print("Missing date token:", hls_path.name)
        continue

    dated_pairs.append({
        "city": city,
        "date": date,
        "hls_path": hls_path,
        "aligned_path": aligned_path,
    })

split_lookup = {}

for city in CITIES:
    city_dates = sorted({
        x["date"] for x in dated_pairs
        if x["city"] == city
    })

    if len(city_dates) < 3:
        raise RuntimeError(
            f"{city} has only {len(city_dates)} distinct dates. "
            "At least three are needed for train/val/test."
        )

    shuffled = list(city_dates)
    rng.shuffle(shuffled)

    n = len(shuffled)
    n_test = max(1, round(0.15 * n))
    n_val = max(1, round(0.15 * n))

    if n_test + n_val >= n:
        n_test, n_val = 1, 1

    test_dates = shuffled[:n_test]
    val_dates = shuffled[n_test:n_test + n_val]
    train_dates = shuffled[n_test + n_val:]

    for d in train_dates:
        split_lookup[(city, d)] = "train"

    for d in val_dates:
        split_lookup[(city, d)] = "val"

    for d in test_dates:
        split_lookup[(city, d)] = "test"

for item in dated_pairs:
    item["split"] = split_lookup[(item["city"], item["date"])]

split_report = pd.DataFrame([
    {
        "city": x["city"],
        "date": x["date"],
        "split": x["split"],
        "hls": x["hls_path"].name,
    }
    for x in dated_pairs
])

display(pd.crosstab(split_report["city"], split_report["split"]))

split_report.to_csv(
    EXPERIMENT_ROOT / "date_split_manifest.csv",
    index=False
)

In [ ]:
from pathlib import Path
import yaml
import numpy as np

# Google Drive project paths
WORKSPACE = Path("/content/drive/MyDrive/colab_workspace")
MODEL_DIR = WORKSPACE / "models_and_configs"

# Original IBM model files
ORIGINAL_CONFIG = MODEL_DIR / "srinagar_fit_config.yaml"
CHECKPOINT = MODEL_DIR / "LST_model.ckpt"

# The config used by the current nine-city experiment.
# Update this path if your notebook saves it under a different name.
FIT_CONFIG = MODEL_DIR / "nine_city_fit_config.yaml"

# Check that the original IBM files exist
assert ORIGINAL_CONFIG.exists(), f"Missing original config: {ORIGINAL_CONFIG}"
assert CHECKPOINT.exists(), f"Missing IBM checkpoint: {CHECKPOINT}"

# Load the original IBM configuration
with open(ORIGINAL_CONFIG, "r") as f:
    original_cfg = yaml.safe_load(f)

# Extract the original data configuration
data_original = original_cfg["data"]["init_args"]

# Extract the original IBM normalisation values
means = np.asarray(data_original["means"], dtype=np.float32)
stds = np.asarray(data_original["stds"], dtype=np.float32)

assert len(means) == 7, f"Expected 7 band means, found {len(means)}"
assert len(stds) == 7, f"Expected 7 band stds, found {len(stds)}"

print("Initialisation successful.")
print("Original config:", ORIGINAL_CONFIG)
print("IBM checkpoint:", CHECKPOINT)
print("Means:", means)
print("Stds:", stds)
print("FIT_CONFIG:", FIT_CONFIG)

Initialisation successful.
Original config: /content/drive/MyDrive/colab_workspace/models_and_configs/srinagar_fit_config.yaml
IBM checkpoint: /content/drive/MyDrive/colab_workspace/models_and_configs/LST_model.ckpt
Means: [ 702.4754  1023.2329  1118.8925  2440.7507  2052.7058  1514.1509
   21.03192]
Stds: [554.82556 613.5565  745.92944 715.0111  761.4761  734.99194   8.66782]
FIT_CONFIG: /content/drive/MyDrive/colab_workspace/models_and_configs/nine_city_fit_config.yaml


In [ ]:
# NINE-CITY PATCH EXTRACTION

from rasterio.windows import Window
import random

PATCH_SIZE = 224
MAX_TRAIN_PATCHES_PER_SCENE = 48
MAX_VAL_PATCHES_PER_SCENE = 12
MAX_TEST_PATCHES_PER_SCENE = 12

MIN_INPUT_VALID_FRACTION = 0.90

random.seed(SEED)
np.random.seed(SEED)

# These come from the original IBM config downloaded in the pilot.
means = np.asarray(data_original["means"], dtype=np.float32)
stds = np.asarray(data_original["stds"], dtype=np.float32)

assert len(means) == 7 and len(stds) == 7

for folder in [
    TRAIN_INPUT, TRAIN_TARGET,
    VAL_INPUT, VAL_TARGET,
    TEST_INPUT, TEST_TARGET
]:
    folder.mkdir(parents=True, exist_ok=True)

patch_manifest = []

split_dirs = {
    "train": (TRAIN_INPUT, TRAIN_TARGET),
    "val": (VAL_INPUT, VAL_TARGET),
    "test": (TEST_INPUT, TEST_TARGET),
}

split_caps = {
    "train": MAX_TRAIN_PATCHES_PER_SCENE,
    "val": MAX_VAL_PATCHES_PER_SCENE,
    "test": MAX_TEST_PATCHES_PER_SCENE,
}


def valid_input_mask(src, image, masks):
    valid_per_band = (
        np.isfinite(image)
        & (masks > 0)
    )

    if src.nodata is not None:
        valid_per_band &= image != src.nodata

    valid_per_band &= image != NODATA

    return valid_per_band


for item in tqdm(dated_pairs, desc="Extracting patches"):

    city = item["city"]
    date = item["date"]
    split = item["split"]

    hls_path = item["hls_path"]
    aligned_path = item["aligned_path"]

    image_dir, target_dir = split_dirs[split]
    cap = split_caps[split]

    with rasterio.open(hls_path) as src, rasterio.open(aligned_path) as tgt:

        if src.count != 7 or tgt.count != 1:
            raise ValueError(f"Unexpected raster bands: {hls_path.name}")

        if (
            src.width != tgt.width
            or src.height != tgt.height
            or src.crs != tgt.crs
            or src.transform != tgt.transform
        ):
            raise ValueError(
                f"HLS/GT grids do not match after alignment: {hls_path.name}"
            )

        H, W = src.height, src.width

        # First pass: store only valid patch coordinates.
        candidates = []

        for row in range(0, H - PATCH_SIZE + 1, PATCH_SIZE):
            for col in range(0, W - PATCH_SIZE + 1, PATCH_SIZE):

                win = Window(col, row, PATCH_SIZE, PATCH_SIZE)

                image = src.read(window=win).astype(np.float32)
                target = tgt.read(1, window=win).astype(np.float32)

                input_masks = src.read_masks(window=win)
                target_mask = tgt.read_masks(1, window=win)

                input_valid_per_band = valid_input_mask(
                    src, image, input_masks
                )
                input_valid = np.all(input_valid_per_band, axis=0)

                target_valid = (
                    np.isfinite(target)
                    & (target_mask > 0)
                    & (target != NODATA)
                )

                if tgt.nodata is not None:
                    target_valid &= target != tgt.nodata

                if input_valid.mean() < MIN_INPUT_VALID_FRACTION:
                    continue

                # Keep only patches with complete GT labels, as in the pilot.
                if not target_valid.all():
                    continue

                candidates.append((row, col))

        random.shuffle(candidates)
        candidates = candidates[:cap]

        # Second pass: write only selected patches.
        for row, col in candidates:

            win = Window(col, row, PATCH_SIZE, PATCH_SIZE)

            image = src.read(window=win).astype(np.float32)
            target = tgt.read(1, window=win).astype(np.float32)

            input_masks = src.read_masks(window=win)
            input_valid_per_band = valid_input_mask(
                src, image, input_masks
            )

            # Fill invalid input values with IBM band means.
            # IBM normalization then maps these to zero.
            for band in range(7):
                image[band][~input_valid_per_band[band]] = means[band]

            # Unique patch IDs across all nine cities.
            patch_id = (
                f"{city}_{hls_path.stem}_r{row}_c{col}"
            )

            image_path = image_dir / f"{patch_id}.inputs.tif"
            label_path = target_dir / f"{patch_id}.lst.tif"

            image_profile = src.profile.copy()
            image_profile.update(
                driver="GTiff",
                height=PATCH_SIZE,
                width=PATCH_SIZE,
                count=7,
                dtype="float32",
                nodata=None,
                compress="deflate",
                predictor=3,
                transform=src.window_transform(win),
            )
            image_profile.pop("blockxsize", None)
            image_profile.pop("blockysize", None)

            label_profile = image_profile.copy()
            label_profile.update(
                count=1,
                dtype="float32",
                nodata=NODATA,
            )

            with rasterio.open(image_path, "w", **image_profile) as dst:
                dst.write(image)

            with rasterio.open(label_path, "w", **label_profile) as dst:
                dst.write(target, 1)

            patch_manifest.append({
                "city": city,
                "date": date,
                "source_scene": hls_path.name,
                "split": split,
                "patch_id": patch_id,
                "input_path": str(image_path),
                "target_path": str(label_path),
                "row": row,
                "col": col,
            })

patch_df = pd.DataFrame(patch_manifest)

if patch_df.empty:
    raise RuntimeError("No patches were generated.")

patch_df.to_csv(
    EXPERIMENT_ROOT / "patch_manifest.csv",
    index=False
)

print("Total patches:", len(patch_df))
display(pd.crosstab(patch_df["city"], patch_df["split"]))

In [ ]:
# DATASET VALIDATION

def verify_patch_pairs(image_dir, label_dir):
    images = {p.name for p in image_dir.glob("*.inputs.tif")}
    labels = {p.name for p in label_dir.glob("*.lst.tif")}

    image_ids = {
        name.replace(".inputs.tif", "")
        for name in images
    }
    label_ids = {
        name.replace(".lst.tif", "")
        for name in labels
    }

    if image_ids != label_ids:
        raise RuntimeError(
            f"Input/label mismatch in {image_dir}\n"
            f"Inputs without labels: {list(image_ids-label_ids)[:5]}\n"
            f"Labels without inputs: {list(label_ids-image_ids)[:5]}"
        )

    return len(image_ids)


for split, (image_dir, label_dir) in split_dirs.items():
    count = verify_patch_pairs(image_dir, label_dir)
    print(f"{split}: {count} paired patches")

print("\nPatch counts by city and split:")
display(pd.crosstab(patch_df.city, patch_df.split))

for split in ["train", "val"]:
    represented = set(patch_df.loc[patch_df.split == split, "city"])
    missing_cities = sorted(set(CITIES) - represented)

    if missing_cities:
        raise RuntimeError(
            f"These cities have no {split} patches: {missing_cities}. "
            "Inspect their scene quality and patch coverage."
        )

# The test set is kept separate and is not used for model selection.
test_count = len(list(TEST_INPUT.glob("*.inputs.tif")))

if test_count == 0:
    raise RuntimeError("No test patches were generated.")

print("Held-out test patches:", test_count)

In [ ]:
# BAND 7 CHECK — CONFIRM AGAINST YOUR DATA PREPARATION METADATA

BAND7_CONFIRMED = False  # Change to True only after verifying the source.

if not BAND7_CONFIRMED:
    raise RuntimeError(
        "STOP: confirm that HLS input band 7 is ERA5-Land 2 m air "
        "temperature and uses the units/scaling expected by IBM. "
        "Then set BAND7_CONFIRMED = True."
    )

print("Band 7 confirmed.")

In [ ]:
# BUILD MULTI-CITY CONFIG FROM THE OFFICIAL IBM CONFIG

import copy
import yaml

cfg = copy.deepcopy(original_cfg)
data_cfg = cfg["data"]["init_args"]

# Train and validation data directories
data_cfg["train_data_root"] = str(TRAIN_INPUT)
data_cfg["train_label_data_root"] = str(TRAIN_TARGET)

data_cfg["val_data_root"] = str(VAL_INPUT)
data_cfg["val_label_data_root"] = str(VAL_TARGET)

# IBM pilot naming convention
data_cfg["img_grep"] = "*.inputs.tif"
data_cfg["label_grep"] = "*.lst.tif"

# Conservative batch settings for Colab GPU
data_cfg["batch_size"] = 1
data_cfg["num_workers"] = 2
data_cfg["drop_last"] = False

# Keep original IBM bands and normalization.
assert len(data_cfg["dataset_bands"]) == 7
assert len(data_cfg["output_bands"]) == 7
assert len(data_cfg["means"]) == 7
assert len(data_cfg["stds"]) == 7

trainer_cfg = cfg["trainer"]

trainer_cfg["accelerator"] = "gpu"
trainer_cfg["devices"] = 1
trainer_cfg["precision"] = "16-mixed"
trainer_cfg["max_epochs"] = 15
trainer_cfg["check_val_every_n_epoch"] = 1
trainer_cfg["num_sanity_val_steps"] = 1
trainer_cfg["log_every_n_steps"] = 10
trainer_cfg["default_root_dir"] = str(RUN_DIR)
trainer_cfg["enable_checkpointing"] = True

# Select the best checkpoint by validation loss and stop if it stops improving.
trainer_cfg["callbacks"] = [
    {
        "class_path": "lightning.pytorch.callbacks.ModelCheckpoint",
        "init_args": {
            "dirpath": str(CHECKPOINT_DIR),
            "filename": "nine-city-best-{epoch:02d}",
            "monitor": "val/loss",
            "mode": "min",
            "save_top_k": 1,
            "save_last": True,
        },
    },
    {
        "class_path": "lightning.pytorch.callbacks.EarlyStopping",
        "init_args": {
            "monitor": "val/loss",
            "mode": "min",
            "patience": 4,
        },
    },
]

logger_cfg = trainer_cfg.get("logger")

if isinstance(logger_cfg, dict):
    init_args = logger_cfg.get("init_args", {})
    if isinstance(init_args, dict):
        init_args["save_dir"] = str(LOG_DIR)
        init_args["name"] = "nine_city_lst"

# Conservative learning rate for fine-tuning.
if "optimizer" in cfg and isinstance(cfg["optimizer"], dict):
    optimizer_args = cfg["optimizer"].get("init_args", {})
    if "lr" in optimizer_args:
        optimizer_args["lr"] = 1e-5

# Preserve IBM's scheduler and other model settings.
if "lr_scheduler" in cfg and isinstance(cfg["lr_scheduler"], dict):
    scheduler_args = cfg["lr_scheduler"].get("init_args", {})
    if "monitor" in scheduler_args:
        scheduler_args["monitor"] = "val/loss"

with open(FIT_CONFIG, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Config saved:", FIT_CONFIG)
print("Base IBM checkpoint:", CHECKPOINT)
print("Train patches:", len(list(TRAIN_INPUT.glob("*.inputs.tif"))))
print("Validation patches:", len(list(VAL_INPUT.glob("*.inputs.tif"))))
print("Held-out test patches:", len(list(TEST_INPUT.glob("*.inputs.tif"))))

# **ONE BATCH SMOKE TEST**

In [ ]:
# TERRATORCH ONE-BATCH SMOKE TEST

import subprocess
import torch

assert torch.cuda.is_available(), "Select a Colab GPU runtime."
assert CHECKPOINT.exists(), f"Missing IBM checkpoint: {CHECKPOINT}"
assert FIT_CONFIG.exists(), f"Missing config: {FIT_CONFIG}"

cmd = [
    "terratorch", "fit",
    "-c", str(FIT_CONFIG),
    "--ckpt_path", str(CHECKPOINT),
    "--trainer.fast_dev_run", "true",
]

print("Running smoke test:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(
        "Smoke test failed. Do not launch full training. "
        "Use the first traceback in the output to diagnose the issue."
    )

print("SMOKE TEST PASSED.")

In [ ]:
# # NINE-CITY FINE-TUNING

# cmd = [
#     "terratorch", "fit",
#     "-c", str(FIT_CONFIG),
#     "--ckpt_path", str(CHECKPOINT),
# ]

# print("Starting nine-city fine-tuning")
# print("Original IBM checkpoint:", CHECKPOINT)
# print("Training patches:", len(list(TRAIN_INPUT.glob("*.inputs.tif"))))
# print("Validation patches:", len(list(VAL_INPUT.glob("*.inputs.tif"))))
# print("Checkpoint output:", CHECKPOINT_DIR)

# result = subprocess.run(
#     cmd,
#     text=True,
#     stdout=subprocess.PIPE,
#     stderr=subprocess.STDOUT,
# )

# print(result.stdout)

# if result.returncode != 0:
#     raise RuntimeError(
#         "Nine-city fine-tuning failed. Preserve the output and logs "
#         "so the first actual error can be diagnosed."
#     )

# print("\nTraining completed.")
# print("Saved checkpoints:")

# for p in sorted(CHECKPOINT_DIR.glob("*.ckpt")):
#     print(p.name, f"{p.stat().st_size / 1e6:.1f} MB")

## 3. Inspect the source CSV and locate the five missing cities

The current CSV is known to contain only four cities. This cell lists exact-named folders under `data/`. It does not assume where the GT folders live.

In [ ]:
# audit=pd.read_csv(SOURCE_CSV)
# assert {'city','granule_id','hls_path','gt_path'}.issubset(audit.columns)
# print(audit.groupby('city').size())
# print('Missing:',sorted(set(CITIES)-set(audit.city)))
# for city in CITIES:
#     found=[p for p in DATA.rglob('*') if p.is_dir() and p.name.lower()==city]
#     print('\\n',city, 'folders:',len(found))
#     for p in found[:15]: print(' ',p)

## 4. Enter the actual HLS and GT folders for missing cities

Fill the four `None` values for each missing city using the folder listing above. Srinagar paths are prefilled from the previously inspected workspace. Do not guess these paths. The existing four-city CSV rows are retained.

In [ ]:
# CITY_DIRS={
#  'jaipur_ind': {
#    'hls':DATA/'hls'/'jaipur_ind'/'hls-predict-ready',
#    'gt':DATA/'jaipur_aligned'/'jaipur_ind'/'target-lst'},
#  'mysuru_ind': {
#    'hls':DATA/'hls'/'mysuru_ind'/'hls-predict-ready',
#    'gt':DATA/'mysuru_aligned'/'mysuru_ind'/'target-lst'},
#  'srinagar_ind': {
#    'hls':DATA/'hls'/'srinagar_ind'/'hls-predict-ready',
#    'gt':DATA/'srinagar_aligned'/'srinagar_ind'/'target-lst'},
#  'surat_ind': {
#    'hls':DATA/'hls'/'surat_ind'/'hls-predict-ready',
#    'gt':DATA/'surat_aligned'/'surat_ind'/'target-lst'},
#  'visakhapatnam_ind': {
#    'hls':DATA/'hls'/'visakhapatnam_ind'/'hls-predict-ready',
#    'gt':DATA/'visakhapatnam_aligned'/'visakhapatnam_ind'/'target-lst'},
# }
# # Replace None with actual Path objects, e.g. DATA/'hls'/'mysuru_ind'/'hls-predict-ready'

## 5. Match HLS and GT scenes

Matching uses shared granule/date tokens and refuses ambiguous matches. Inspect the report. If filenames do not share tokens, create explicit pairs rather than weakening the matcher.

In [ ]:
# def tifs(folder):
#     p=Path(folder)
#     return sorted(list(p.rglob('*.tif'))+list(p.rglob('*.tiff'))) if p.exists() else []
# def tokens(path):
#     s=Path(path).stem.upper()
#     out=set(re.findall(r'T\\d{2}[A-Z]{3}\\.20\\d{6}',s))
#     out.update(re.findall(r'20\\d{6}',s))
#     return out

# new=[]
# for city,dirs in CITY_DIRS.items():
#     if dirs['hls'] is None or dirs['gt'] is None:
#         print('PATHS REQUIRED:',city); continue
#     hs,gs=tifs(dirs['hls']),tifs(dirs['gt'])
#     used=set(); ambiguous=[]; unmatched=[]
#     for h in hs:
#         ht=tokens(h)
#         scores=[(len(ht & tokens(g)),g,ht & tokens(g)) for g in gs]
#         scores=[x for x in scores if x[0]>0]
#         if not scores: unmatched.append(str(h)); continue
#         top=max(x[0] for x in scores)
#         winners=[x for x in scores if x[0]==top]
#         if len(winners)!=1 or str(winners[0][1]) in used:
#             ambiguous.append(str(h)); continue
#         _,g,common=winners[0]; used.add(str(g))
#         granule=sorted(common,key=len)[-1]
#         new.append({'city':city,'granule_id':granule,'hls_path':str(h),'gt_path':str(g)})
#     print(city, 'HLS',len(hs),'GT',len(gs),'matched',sum(x['city']==city for x in new),
#           'unmatched HLS',len(unmatched),'ambiguous',len(ambiguous))

# extra=pd.DataFrame(new)
# combined=pd.concat([audit,extra],ignore_index=True) if len(extra) else audit.copy()
# combined=combined.drop_duplicates(['city','hls_path','gt_path'])
# missing=sorted(set(CITIES)-set(combined.city))
# print('City counts:'); display(combined.groupby('city').size())
# if missing:
#     raise RuntimeError('Missing city scene pairs: '+str(missing)+'. Fill CITY_DIRS and rerun from this cell.')

## 6. Align HLS to each GT grid and audit quality

In [ ]:
# import rasterio
# from rasterio.warp import reproject,Resampling

# def valid_gt(ds,a):
#     m=(ds.read_masks(1)>0)&np.isfinite(a)&(a>-9000)
#     if ds.nodata is not None: m &= a!=ds.nodata
#     return m

# qrows=[]; aligned=[]
# for i,r in combined[combined.city.isin(CITIES)].reset_index(drop=True).iterrows():
#     try:
#         with rasterio.open(r.hls_path) as hs, rasterio.open(r.gt_path) as gs:
#             if hs.count<7: raise ValueError(f'HLS has {hs.count} bands, expected 7')
#             y=gs.read(1).astype('float32'); ym=valid_gt(gs,y)
#             if not ym.any(): raise ValueError('GT has no valid pixels')
#             vals=y[ym]; p01,p50,p99=np.percentile(vals,[1,50,99])
#             if p01<0 or p99>65:
#                 warnings.warn(f'Suspicious GT Celsius range {r.city}/{r.granule_id}: {p01:.2f}, {p99:.2f}')
#             X=np.full((7,gs.height,gs.width),np.nan,np.float32)
#             for b in range(1,8):
#                 reproject(source=rasterio.band(hs,b),destination=X[b-1],
#                     src_transform=hs.transform,src_crs=hs.crs,
#                     dst_transform=gs.transform,dst_crs=gs.crs,
#                     src_nodata=hs.nodata,dst_nodata=np.nan,
#                     resampling=Resampling.bilinear)
#             valid=ym & np.all(np.isfinite(X),axis=0)
#             frac=float(valid.mean())
#             qrows.append({'city':r.city,'granule_id':r.granule_id,'joint_valid_fraction':frac,
#                           'gt_p01_C':p01,'gt_p50_C':p50,'gt_p99_C':p99})
#             if frac>=MIN_SCENE_VALID:
#                 f=ALIGNED/r.city/f'{i:05d}.npz'; f.parent.mkdir(parents=True,exist_ok=True)
#                 np.savez_compressed(f,X=X,y=y,valid=valid)
#                 aligned.append({'city':r.city,'granule_id':r.granule_id,'hls_path':r.hls_path,
#                                 'aligned_path':str(f)})
#     except Exception as e:
#         qrows.append({'city':r.city,'granule_id':r.granule_id,'error':repr(e)})

# quality=pd.DataFrame(qrows); manifest=pd.DataFrame(aligned)
# quality.to_csv(OUT/'quality_audit.csv',index=False)
# manifest.to_csv(OUT/'aligned_manifest.csv',index=False)
# display(quality.groupby('city').agg(scenes=('granule_id','count'),
#         median_valid=('joint_valid_fraction','median')))
# if set(manifest.city)!=set(CITIES):
#     raise RuntimeError('One or more cities have no accepted aligned scenes. Inspect quality_audit.csv.')

## 7. Date-based split within each city

In [ ]:
# def get_date(granule,path):
#     m=re.search(r'(20\\d{6})',str(granule)+' '+Path(path).name)
#     return m.group(1) if m else None
# manifest['date']=manifest.apply(lambda r:get_date(r.granule_id,r.hls_path),axis=1)
# if manifest.date.isna().any():
#     raise RuntimeError('Some scenes have no YYYYMMDD date token. Resolve dates before splitting.')
# rng=np.random.default_rng(SEED); split={}
# for city,g in manifest.groupby('city'):
#     dates=np.array(sorted(g.date.unique())); rng.shuffle(dates)
#     if len(dates)<3: raise RuntimeError(f'{city} has fewer than 3 distinct dates.')
#     nt=max(1,round(.15*len(dates))); nv=max(1,round(.15*len(dates)))
#     if nt+nv>=len(dates): nt,nv=1,1
#     for d in dates[nt+nv:]: split[(city,d)]='train'
#     for d in dates[nt:nt+nv]: split[(city,d)]='val'
#     for d in dates[:nt]: split[(city,d)]='test'
# manifest['split']=manifest.apply(lambda r:split[(r.city,r.date)],axis=1)
# display(pd.crosstab(manifest.city,manifest.split))
# manifest.to_csv(OUT/'aligned_manifest.csv',index=False)

## 8. Extract 128×128 patches and retain the mask

In [ ]:
# patch_rows=[]
# for _,r in manifest.iterrows():
#     z=np.load(r.aligned_path); X=z['X'].astype('float32'); y=z['y'].astype('float32'); v=z['valid'].astype(bool)
#     H,W=y.shape
#     for top in range(0,H-PATCH+1,STRIDE):
#       for left in range(0,W-PATCH+1,STRIDE):
#         sl=np.s_[top:top+PATCH,left:left+PATCH]
#         m=v[sl]&np.all(np.isfinite(X[:,sl[0],sl[1]]),axis=0)
#         if m.mean()<MIN_PATCH_VALID: continue
#         xx=X[:,sl[0],sl[1]].copy(); yy=y[sl].copy()
#         xx[:,~m]=0; yy[~m]=0
#         f=PATCHES/f'{r.city}_{r.date}_{len(patch_rows):08d}.npz'
#         np.savez_compressed(f,X=xx,y=yy,mask=m)
#         patch_rows.append({'city':r.city,'date':r.date,'split':r.split,'patch_file':str(f)})
# pm=pd.DataFrame(patch_rows); pm.to_csv(OUT/'patch_manifest.csv',index=False)
# display(pd.crosstab(pm.city,pm.split))
# if pm.empty or not {'train','val','test'}.issubset(set(pm.split)):
#     raise RuntimeError('Patch extraction did not produce all three splits.')

## 9. Training-only normalization (targets stay in °C)

In [ ]:
# sx=np.zeros(7); sx2=np.zeros(7); cnt=np.zeros(7); target=[]
# for _,r in pm[pm.split=='train'].iterrows():
#     z=np.load(r.patch_file); x=z['X'].astype('float64'); m=z['mask'].astype(bool)
#     for b in range(7):
#         v=x[b][m]; sx[b]+=v.sum(); sx2[b]+=(v*v).sum(); cnt[b]+=v.size
#     target.append(z['y'][m])
# if np.any(cnt==0): raise RuntimeError('Empty training band.')
# means=sx/cnt; stds=np.sqrt(np.maximum(sx2/cnt-means**2,1e-8))
# stats={'input_means':means.tolist(),'input_stds':stds.tolist(),
#        'target_unit':'C','target_normalization':'none'}
# json.dump(stats,open(OUT/'normalization_stats.json','w'),indent=2)
# print(stats)

## 10. PyTorch datasets and city-balanced training sampler

In [ ]:
# import torch
# from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
# class PatchDS(Dataset):
#     def __init__(self,frame): self.f=frame.reset_index(drop=True)
#     def __len__(self): return len(self.f)
#     def __getitem__(self,i):
#         r=self.f.iloc[i]; z=np.load(r.patch_file)
#         x=(z['X'].astype('float32')-means[:,None,None])/np.maximum(stds[:,None,None],1e-6)
#         y=z['y'].astype('float32'); m=z['mask'].astype(bool)
#         x[:,~m]=0; y[~m]=0
#         return torch.from_numpy(x),torch.from_numpy(y[None]),torch.from_numpy(m[None].astype('float32'))

# tr=pm[pm.split=='train']; va=pm[pm.split=='val']; te=pm[pm.split=='test']
# train_ds,val_ds,test_ds=PatchDS(tr),PatchDS(va),PatchDS(te)
# counts=tr.city.value_counts().to_dict()
# w=tr.city.map(lambda c:1/counts[c]).to_numpy()
# sampler=WeightedRandomSampler(torch.as_tensor(w,dtype=torch.double),len(w),replacement=True)
# BATCH=2
# train_loader=DataLoader(train_ds,batch_size=BATCH,sampler=sampler,num_workers=2,pin_memory=True)
# val_loader=DataLoader(val_ds,batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True)
# test_loader=DataLoader(test_ds,batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True)
# bx,by,bm=next(iter(train_loader))
# print(bx.shape,by.shape,bm.shape)
# assert bx.shape[1:]==(7,PATCH,PATCH)

## 11. Build the original model architecture and strictly verify weight compatibility

Known prior configuration: `prithvi_swin_L`, seven input channels, `UperNetDecoder`. This section is deliberately fail-fast. If the installed TerraTorch version cannot build that exact architecture, or if checkpoint coverage is low, stop and reconcile the model definition before training. Do not silently substitute a different backbone.

In [ ]:
# from terratorch.models import EncoderDecoderFactory
# factory=EncoderDecoderFactory()
# MODEL_ARGS={
#  'backbone':'prithvi_swin_L',
#  'backbone_pretrained':False,
#  'backbone_num_frames':1,
#  'backbone_bands':['BLUE','GREEN','RED','NIR_NARROW','SWIR_1','SWIR_2','7'],
#  'decoder':'UperNetDecoder',
#  'in_channels':7,
#  'num_outputs':1,
# }
# model=factory.build_model(task='regression',**MODEL_ARGS)

# raw=torch.load(ORIGINAL_WEIGHTS,map_location='cpu',weights_only=False)
# src=raw
# if isinstance(raw,dict):
#     for k in ['state_dict','model_state_dict']:
#         if k in raw and isinstance(raw[k],dict): src=raw[k]; break
# target=model.state_dict()
# matched={k:v for k,v in src.items() if k in target and torch.is_tensor(v) and target[k].shape==v.shape}
# coverage=sum(target[k].numel() for k in matched)/max(sum(v.numel() for v in target.values()),1)
# print('matched',len(matched),'of',len(target),'state tensors; parameter coverage',coverage)
# if coverage<0.80:
#     raise RuntimeError('Checkpoint coverage <80%. Architecture does not match original weights; do not train.')
# missing,unexpected=model.load_state_dict(matched,strict=False)
# print('Missing keys:',missing[:30])

## 12. One-batch forward check

In [ ]:
# device=torch.device('cuda')
# model=model.to(device)
# bx,by,bm=next(iter(train_loader)); bx,by,bm=bx.to(device),by.to(device),bm.to(device)
# def get_pred(x):
#     o=model(x)
#     if hasattr(o,'output'): o=o.output
#     elif isinstance(o,dict): o=o.get('output',o.get('pred'))
#     if isinstance(o,(tuple,list)): o=o[0]
#     if not torch.is_tensor(o): raise TypeError(f'Unexpected model output: {type(o)}')
#     if o.ndim==3: o=o[:,None]
#     if o.shape[-2:]!=x.shape[-2:]:
#         o=torch.nn.functional.interpolate(o,size=x.shape[-2:],mode='bilinear',align_corners=False)
#     return o
# with torch.no_grad():
#     p=get_pred(bx)
#     loss=torch.sqrt((((p.float()-by.float())**2)*bm).sum()/bm.sum().clamp_min(1))
# print('Output:',p.shape,'masked RMSE:',float(loss))
# assert p.shape==by.shape and torch.isfinite(loss)

## 13. Conservative fine-tuning with validation early stopping

In [ ]:
# from torch.optim import AdamW
# from torch.optim.lr_scheduler import ReduceLROnPlateau

# def evaluate(loader):
#     model.eval(); sse=n=0.
#     with torch.no_grad():
#       for x,y,m in loader:
#         x,y,m=x.to(device),y.to(device),m.to(device)
#         p=get_pred(x).float()
#         sse+=float((((p-y)**2)*m).sum()); n+=float(m.sum())
#     return math.sqrt(sse/max(n,1))

# optimizer=AdamW(model.parameters(),lr=1e-5,weight_decay=0.05)
# scheduler=ReduceLROnPlateau(optimizer,mode='min',factor=.5,patience=2)
# MAX_EPOCHS=20; PATIENCE=5; best=float('inf'); bad=0; history=[]
# for epoch in range(1,MAX_EPOCHS+1):
#     model.train(); sse=n=0.
#     for x,y,m in train_loader:
#         x,y,m=x.to(device),y.to(device),m.to(device)
#         optimizer.zero_grad(set_to_none=True)
#         p=get_pred(x)
#         loss=torch.sqrt((((p.float()-y)**2)*m).sum()/m.sum().clamp_min(1))
#         if not torch.isfinite(loss): raise RuntimeError('Non-finite training loss.')
#         loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
#         sse+=float((((p.detach().float()-y)**2)*m).sum()); n+=float(m.sum())
#     trm=math.sqrt(sse/max(n,1)); vam=evaluate(val_loader); scheduler.step(vam)
#     history.append({'epoch':epoch,'train_RMSE_C':trm,'val_RMSE_C':vam})
#     print(f'Epoch {epoch}: train {trm:.4f} C | val {vam:.4f} C')
#     if vam<best:
#         best=vam; bad=0
#         torch.save({'state_dict':model.state_dict(),'epoch':epoch,'val_RMSE_C':best,'stats':stats},
#                    CKPTS/'best_nine_city_model.pt')
#     else:
#         bad+=1
#         if bad>=PATIENCE: print('Early stopping'); break
# pd.DataFrame(history).to_csv(OUT/'training_history.csv',index=False)
# best_ckpt=torch.load(CKPTS/'best_nine_city_model.pt',map_location=device,weights_only=False)
# model.load_state_dict(best_ckpt['state_dict'])

## 14. Test metrics and per-city results

In [ ]:
# def metrics(loader):
#     model.eval(); ys=[]; ps=[]
#     with torch.no_grad():
#       for x,y,m in loader:
#         x,y,m=x.to(device),y.to(device),m.to(device)
#         p=get_pred(x).float(); v=m.bool()
#         ys.append(y[v].cpu().numpy()); ps.append(p[v].cpu().numpy())
#     y=np.concatenate(ys); p=np.concatenate(ps); e=p-y
#     return {'MAE_C':float(np.mean(np.abs(e))),
#             'RMSE_C':float(np.sqrt(np.mean(e**2))),
#             'Bias_C':float(e.mean()),
#             'R2':float(1-(e**2).sum()/((y-y.mean())**2).sum()),
#             'n_pixels':int(y.size)}
# finetuned=metrics(test_loader)
# # Evaluate baseline with the same exact architecture and original weights.
# baseline=factory.build_model(task='regression',**MODEL_ARGS)
# bt=baseline.state_dict()
# bm={k:v for k,v in src.items() if k in bt and torch.is_tensor(v) and bt[k].shape==v.shape}
# bcov=sum(bt[k].numel() for k in bm)/max(sum(v.numel() for v in bt.values()),1)
# if bcov<.80: raise RuntimeError('Baseline weights do not match model.')
# baseline.load_state_dict(bm,strict=False)
# trained=model; model=baseline.to(device); base_metrics=metrics(test_loader); model=trained
# comparison=pd.DataFrame([{'model':'Original IBM',**base_metrics},
#                          {'model':'Nine-city fine-tuned',**finetuned}])
# display(comparison)
# comparison.to_csv(OUT/'baseline_vs_finetuned_test.csv',index=False)

# # Per-city test evaluation
# city_rows=[]
# for city in CITIES:
#     f=te[te.city==city]
#     if f.empty:
#         city_rows.append({'city':city,'error':'no test patches'}); continue
#     ds=PatchDS(f); dl=DataLoader(ds,batch_size=BATCH,shuffle=False,num_workers=2)
#     city_rows.append({'city':city,**metrics(dl)})
# city_table=pd.DataFrame(city_rows)
# display(city_table)
# city_table.to_csv(OUT/'per_city_test_metrics.csv',index=False)
# print('Artifacts saved in:',OUT)

## 15. How to interpret the result

- Claim improvement only if held-out test MAE/RMSE improve over the original IBM baseline.
- Inspect per-city results. Pooled improvement can hide degradation in one or more cities.
- This notebook's date split evaluates temporal generalization **within the nine represented cities**. It does not prove transfer to unseen cities. For that, run a separate leave-one-city-out experiment.
- If the original checkpoint match or forward test fails, stop. Do not lower the compatibility threshold to force training.
- GT Celsius outliers and QA issues must be resolved at source; do not clip them just to improve metrics.